# Análise Exploratória — Indicium Price Predictor

Anúncios de aluguel temporário em Nova York. O objetivo aqui é entender o que
move o preço antes de treinar o modelo: onde ficam os anúncios caros, como as
variáveis se relacionam e onde estão os dados faltantes.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Caminho relativo ao notebook, para rodar em qualquer máquina.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CSV_PATH = PROJECT_ROOT / "data" / "teste_indicium_precificacao.csv"

df = pd.read_csv(CSV_PATH)
df.head()

## Informações básicas do dataset

In [ ]:
print("Informações básicas do dataset:")
df.info()

print("\nEstatísticas descritivas:")
df.describe()

## Valores ausentes

`ultima_review` e `reviews_por_mes` faltam juntos — são os anúncios que nunca
receberam avaliação.

In [ ]:
df.isnull().sum()

## Distribuição dos preços

A cauda é longa: a maioria dos anúncios se concentra na faixa baixa e alguns
poucos puxam a escala para a direita.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['price'], bins=50, kde=True, color='blue')
plt.title('Distribuição dos Preços')
plt.xlabel('Preço')
plt.ylabel('Frequência')
plt.show()

## Preço médio por bairro e por região

In [ ]:
avg_price_by_neighborhood = df.groupby('bairro')['price'].mean().sort_values(ascending=False)
avg_price_by_bairro_group = df.groupby('bairro_group')['price'].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(
    x=avg_price_by_neighborhood.index,
    y=avg_price_by_neighborhood.values,
    hue=avg_price_by_neighborhood.index,
    palette='viridis',
    legend=False,
)
plt.xticks(rotation=90)
plt.title('Preço Médio por Bairro')
plt.xlabel('Bairro')
plt.ylabel('Preço Médio')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.barplot(
    x=avg_price_by_bairro_group.index,
    y=avg_price_by_bairro_group.values,
    hue=avg_price_by_bairro_group.index,
    palette='coolwarm',
    legend=False,
)
plt.title('Preço Médio por Bairro Group')
plt.xlabel('Bairro Group')
plt.ylabel('Preço Médio')
plt.show()

## Tipo de quarto e preço

Casa/apartamento inteiro custa bem mais que quarto privado, e quarto
compartilhado fica na faixa mais baixa.

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(data=df, x='room_type', y='price', hue='room_type', palette='Set2', legend=False)
plt.title('Relação entre Tipo de Quarto e Preço')
plt.xlabel('Tipo de Quarto')
plt.ylabel('Preço')
plt.show()

## Correlação entre as variáveis numéricas

Nenhuma variável numérica sozinha explica o preço — o que justifica um modelo
baseado em árvores, que captura combinações entre elas.

In [ ]:
numeric_features = ['latitude', 'longitude', 'price', 'minimo_noites', 'numero_de_reviews',
                    'reviews_por_mes', 'calculado_host_listings_count', 'disponibilidade_365']

correlation_matrix = df[numeric_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matriz de Correlação entre Variáveis Numéricas')
plt.tight_layout()
plt.show()

## Mínimo de noites, disponibilidade e preço

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='minimo_noites', y='price', hue='disponibilidade_365', palette='viridis')
plt.title('Relação entre Mínimo de Noites, Disponibilidade e Preço')
plt.xlabel('Mínimo de Noites')
plt.ylabel('Preço')
plt.show()

## Outliers no preço

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x=df['price'], color='orange')
plt.title('Boxplot dos Preços')
plt.xlabel('Preço')
plt.show()

## Distribuição de reviews por mês

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['reviews_por_mes'].dropna(), bins=50, kde=True, color='green')
plt.title('Distribuição de Reviews por Mês')
plt.xlabel('Reviews por Mês')
plt.ylabel('Frequência')
plt.show()

## Anfitriões com mais anúncios

In [ ]:
top_hosts = df['host_name'].value_counts().head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_hosts.index, y=top_hosts.values, hue=top_hosts.index, palette='Blues_d', legend=False)
plt.title('Top 10 com Mais Listagens')
plt.xlabel('Host Name')
plt.ylabel('Número de Listagens')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Bairros com maior retorno financeiro

Retorno = preço × dias disponíveis no ano. Um bairro caro com pouca
disponibilidade pode render menos que um bairro médio aberto o ano inteiro.

In [ ]:
df['retorno_financeiro'] = df['price'] * df['disponibilidade_365']
retorno_por_bairro = df.groupby('bairro')['retorno_financeiro'].mean().sort_values(ascending=False)
retorno_por_bairro.head(20)